In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from skimage.graph import route_through_array
from matplotlib.collections import LineCollection
from pathlib import Path
import copy
from scipy.ndimage import convolve
from skimage.morphology import binary_dilation

DATA_DIR = Path("/Users/sylvi/topo_data/temp/")
IMAGE_FILE = "20251209_TAF_pICOz_MQ_50_percent_damage.0_00010.topostats"
FILE_PATH = DATA_DIR / IMAGE_FILE
GRAIN_IDX = "1"

height_threshold_nm = 2.2  # Minimum height in nm required for a crossing to be considered a true crossing.

crossing_data = []

with h5py.File(FILE_PATH, "r") as f:
    image = f["image"][:]
    pixel_to_nm_scaling = f["pixel_to_nm_scaling"]
    plt.imshow(image, cmap="afmhot", origin="lower")
    plt.show()
    print(f.keys())
    graincrops = f["grain_crops"]

    for grain_index, graincrop in graincrops.items():
        print(f"grain index: {grain_index}")
        grain_image = graincrop["image"][:]
        bbox = graincrop["bbox"]
        # print(graincrop.keys())
        disordered_trace = graincrop["disordered_trace"]
        # print(disordered_trace.keys())
        disordered_trace_images = disordered_trace["images"]
        # print(disordered_trace_images.keys())
        branch_indexes_image = disordered_trace_images["branch_indexes"][:]
        branch_indexes_nonzero = np.argwhere(branch_indexes_image > 0)
        # print(branch_indexes_nonzero)
        pruned_skeleton = disordered_trace_images["pruned_skeleton"][:].astype(np.int32)

        # find the nodes
        convolved_skeleton = convolve(pruned_skeleton, np.ones((3, 3)))
        convolved_skeleton[pruned_skeleton == 0] = 0

        print(convolved_skeleton.shape)
        nodes = graincrop["nodes"]
        print(f"nodes: {nodes.keys()}")

        for node_index, node in nodes.items():
            print(f"node index: {node_index}")
            print(node.keys())

            node_area_skeleton = node["node_area_skeleton"][:]
            print(np.unique(node_area_skeleton))
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            ax.imshow(node_area_skeleton, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} area skeleton")
            plt.show()

            mask_node = np.where(node_area_skeleton == 3, 1, 0)
            mask_branch = np.where(node_area_skeleton == 1, 1, 0)

            # get the maximum height of any node pixel from the image
            node_pixel_coords = np.argwhere(mask_node == 1)
            node_pixel_heights = grain_image[node_pixel_coords[:, 0], node_pixel_coords[:, 1]]
            max_node_pixel_height = np.max(node_pixel_heights)
            if max_node_pixel_height > height_threshold_nm:
                # This is a true crossing, so we skip it and mark it as such
                print(f"node {node_index} is a true crossing with max height {max_node_pixel_height:.2f} nm, skipping")
                crossing_data.append(
                    {
                        "grain_index": grain_index,
                        "node_index": node_index,
                        "true_crossing": True,
                        "max_node_pixel_height": max_node_pixel_height,
                        "coords": node_pixel_coords,
                        "centroid": np.mean(node_pixel_coords, axis=0),
                        "bbox": bbox,
                    }
                )
                continue
            else:
                crossing_data.append(
                    {
                        "grain_index": grain_index,
                        "node_index": node_index,
                        "true_crossing": False,
                        "max_node_pixel_height": max_node_pixel_height,
                        "coords": node_pixel_coords,
                        "centroid": np.mean(node_pixel_coords, axis=0),
                        "bbox": bbox,
                    }
                )

            mask_dilated_node = binary_dilation(mask_node, footprint=np.ones((3, 3)))
            mask_dilated_node_branch_intersections = mask_dilated_node * mask_branch
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            ax.imshow(mask_dilated_node_branch_intersections, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} dilated node-branch intersections")
            plt.show()

            branch_starts = np.argwhere(mask_dilated_node_branch_intersections == 1)
            if branch_starts.shape[0] != 4:
                print(
                    f"Warning: expected 4 branch starts for node {node_index} but found {branch_starts.shape[0]}, skipping"
                )
                continue

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image, cmap="afmhot", origin="lower")
    for crossing in crossing_data:
        bbox = crossing["bbox"]
        if crossing["true_crossing"]:
            ax.scatter(
                crossing["centroid"][1] + bbox[1],
                crossing["centroid"][0] + bbox[0],
                color="green",
                edgecolors="white",
                s=100,
            )
        else:
            ax.scatter(
                crossing["centroid"][1] + bbox[1],
                crossing["centroid"][0] + bbox[0],
                color="red",
                edgecolors="white",
                s=100,
            )
    plt.legend()
    plt.title("True crossings identified in the image")
    plt.show()